# CLAP Text-to-Music Retrieval Demo

Search for music using natural language queries like "chill lo-fi vibes" or "aggressive heavy metal".

**Prerequisites:** Run `python scripts/generate_clap_embeddings.py` and `python scripts/build_faiss_index.py` first.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import IPython.display as ipd
from src.config import PROCESSED_DIR, FMA_SMALL_DIR
from src.metadata import load_tracks
from src.audio_utils import get_audio_path
from src.embeddings.clap import CLAPEmbeddingGenerator
from src.indexing.faiss_index import FaissIndex

## Load Index and Model

In [ ]:
# Load FAISS index
index = FaissIndex.load(PROCESSED_DIR / 'clap_faiss.index')
print(f'Index loaded: {index.index.ntotal} tracks')

# Load metadata for display
tracks = load_tracks()

# Load CLAP model for text embedding
clap = CLAPEmbeddingGenerator()

## Helper: Display Results

In [ ]:
def search_and_display(query, k=5):
    """Search for music by text query and display results with audio players."""
    text_emb = clap.embed_text([query])
    results = index.query(text_emb[0], k=k)
    
    print(f'Query: "{query}"\n{"=" * 60}')
    for rank, (tid, score) in enumerate(results, 1):
        row = tracks.loc[tid]
        title = row.get(('track', 'title'), 'Unknown')
        artist = row.get(('artist', 'name'), 'Unknown')
        genre = row.get(('track', 'genre_top'), 'Unknown')
        
        print(f'\n#{rank} (score: {score:.4f}) | {title} — {artist} [{genre}]')
        
        audio_path = get_audio_path(tid)
        if audio_path.exists():
            display(ipd.Audio(str(audio_path)))

## Try Some Queries

In [ ]:
search_and_display('chill lo-fi vibes')

In [ ]:
search_and_display('aggressive heavy metal with fast drums')

In [ ]:
search_and_display('upbeat happy pop song')

In [ ]:
search_and_display('sad piano ballad')

In [ ]:
search_and_display('funky bass groove')

## Custom Query
Try your own!

In [ ]:
# search_and_display('your query here', k=10)